# OptiTrack vs Mediapipe tracker comparison

This notebook compares the **large-motion OptiTrack trials** against the Mediapipe/camera tracking session:

`Mediapipe-capture_handtest/2026_09_01_19_21_single_finger_config_fingers_motor_set_0_167`

Main outputs are written to:

`OT-results/tracker_comparison/`

## What is compared

- OptiTrack: filtered marker position from `OT-results/filtered_data/*_filtered_position_mm.csv`
- Camera/Mediapipe: `active_finger_x`, `active_finger_y` from each `pair_XXX/tracking.csv`

Because Mediapipe is a 2D overhead camera tracker, the numeric error here is a **2D plane comparison**:

- OptiTrack plane: **X-Z** in millimeters
- Mediapipe plane: camera image **x-y** in pixels, calibrated into OptiTrack X-Z millimeters

## Important method choices

1. **Timestamp matching**: OptiTrack and Mediapipe are matched by absolute timestamps, not by file order.
2. **Overlap only**: only OptiTrack files that overlap the given Mediapipe capture are compared.
3. **Axis/scale/zero calibration**: for each matched segment, Mediapipe pixels are mapped to OptiTrack X-Z mm using an affine calibration.
4. **Lag search**: a small time-shift search is performed to handle possible clock/recording offset.
5. **Validation metric**: RMSE, mean positional error, and 95th-percentile positional error are computed on cross-validated residuals after calibration.

This estimates tracker agreement after correcting coordinate origin, axis direction, rotation, scale, and mild camera anisotropy. It does **not** measure uncalibrated absolute camera accuracy.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.signal import butter, sosfiltfilt
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    warnings.warn('SciPy unavailable; low-pass smoothing will use rolling mean fallback.')

ROOT = Path('.')
OT_DATA_DIR = ROOT / 'OT-data'
OT_RESULTS_DIR = ROOT / 'OT-results'
OT_FILTERED_DIR = OT_RESULTS_DIR / 'filtered_data'
OT_SUMMARY_CSV = OT_RESULTS_DIR / 'position_summary.csv'
MP_DIR = ROOT / 'Mediapipe-capture_handtest' / '2026_09_01_19_21_single_finger_config_fingers_motor_set_0_167'

OUT_DIR = OT_RESULTS_DIR / 'tracker_comparison'
PLOT_DIR = OUT_DIR / 'plots'
SEGMENT_DIR = OUT_DIR / 'segment_samples'
for d in [OUT_DIR, PLOT_DIR, SEGMENT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MIN_OVERLAP_SECONDS = 20.0
GLOBAL_LAG_SEARCH_SECONDS = np.arange(-120.0, 120.0001, 0.5)
LOCAL_LAG_SEARCH_SECONDS = np.arange(-2.0, 2.0001, 0.02)
MIN_OT_COVERAGE_FRACTION = 0.70
MP_LPF_CUTOFF_HZ = 8.0
CV_FOLDS = 5

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.grid'] = True

In [ ]:
def parse_ot_start_time(csv_path: Path):
    """Parse Motive Capture Start Time from a CSV header."""
    with csv_path.open(newline='', encoding='utf-8-sig') as f:
        first = next(csv.reader(f))
    meta = {}
    for i in range(0, len(first) - 1, 2):
        if first[i]:
            meta[first[i].strip()] = first[i + 1].strip()
    start_str = meta['Capture Start Time']  # e.g. 2026-09-01 07.22.32.420 PM
    start = dt.datetime.strptime(start_str, '%Y-%m-%d %I.%M.%S.%f %p')
    return pd.Timestamp(start), meta


def safe_stem(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', Path(name).stem).strip('_')


def lowpass_array(values, fs, cutoff_hz):
    values = np.asarray(values, dtype=float)
    if SCIPY_AVAILABLE and len(values) > 30 and cutoff_hz < fs / 2:
        sos = butter(4, cutoff_hz / (fs / 2), btype='lowpass', output='sos')
        return sosfiltfilt(sos, values, axis=0)
    window = max(3, int(round(fs / max(cutoff_hz, 0.1))))
    if window % 2 == 0:
        window += 1
    return pd.DataFrame(values).rolling(window, center=True, min_periods=1).mean().to_numpy(dtype=float)


def load_ot_big_trials():
    summary = pd.read_csv(OT_SUMMARY_CSV)
    big = summary[summary['motion_scale'].eq('big')].copy().sort_values('file').reset_index(drop=True)
    rows = []
    for _, row in big.iterrows():
        raw_csv = OT_DATA_DIR / row['file']
        filtered_csv = Path(row['filtered_csv'])
        if not filtered_csv.exists():
            filtered_csv = OT_FILTERED_DIR / f"{safe_stem(row['file'])}_filtered_position_mm.csv"
        start, meta = parse_ot_start_time(raw_csv)
        df = pd.read_csv(filtered_csv)
        t = pd.to_timedelta(df['time_s'].to_numpy(dtype=float), unit='s') + start
        ot = df[['filtered_X_mm', 'filtered_Z_mm']].to_numpy(dtype=float)
        y = df['filtered_Y_mm'].to_numpy(dtype=float)
        rows.append({
            'file': row['file'],
            'marker': row['marker'],
            'start': t[0],
            'end': t[-1],
            'time': pd.to_datetime(t),
            'ot_xz_mm': ot,
            'ot_y_mm': y,
            'filtered_csv': str(filtered_csv),
            'duration_s': float(row['duration_s']),
        })
    return rows


def load_mp_pairs():
    pairs = []
    for tracking_csv in sorted(MP_DIR.glob('pair_*/tracking.csv')):
        df = pd.read_csv(tracking_csv)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp').reset_index(drop=True)
        finger = str(df['finger'].dropna().iloc[0]) if 'finger' in df else tracking_csv.parent.name
        t_seconds = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds().to_numpy(dtype=float)
        fs_est = 1.0 / np.nanmedian(np.diff(t_seconds)) if len(t_seconds) > 2 else 60.0
        mp_xy_raw = df[['active_finger_x', 'active_finger_y']].apply(pd.to_numeric, errors='coerce').interpolate(limit_direction='both').to_numpy(dtype=float)
        mp_xy_smooth = lowpass_array(mp_xy_raw, fs_est, MP_LPF_CUTOFF_HZ)
        pairs.append({
            'pair': tracking_csv.parent.name,
            'finger': finger,
            'start': df['timestamp'].iloc[0],
            'end': df['timestamp'].iloc[-1],
            'time': df['timestamp'],
            'mp_xy_px_raw': mp_xy_raw,
            'mp_xy_px': mp_xy_smooth,
            'tracking_csv': str(tracking_csv),
            'fs_est': fs_est,
            'rows': len(df),
        })
    return pairs


def seconds(ts):
    """Convert pandas datetimes to float seconds relative to Unix epoch.

    Avoid dtype-unit surprises (ns/us) by using Timestamp.value, which is always ns.
    """
    ser = pd.Series(pd.to_datetime(ts))
    return ser.map(lambda x: x.value).to_numpy(dtype=float) / 1e9

def interp_2d(query_s, source_time_s, source_values):
    return np.column_stack([
        np.interp(query_s, source_time_s, source_values[:, 0]),
        np.interp(query_s, source_time_s, source_values[:, 1]),
    ])


def fit_affine(src_xy, dst_xz):
    """Fit [src_x, src_y, 1] @ B = [dst_x, dst_z]."""
    X = np.column_stack([src_xy[:, 0], src_xy[:, 1], np.ones(len(src_xy))])
    B, *_ = np.linalg.lstsq(X, dst_xz, rcond=None)
    return B


def apply_affine(src_xy, B):
    X = np.column_stack([src_xy[:, 0], src_xy[:, 1], np.ones(len(src_xy))])
    return X @ B


def errors_from_predictions(pred, truth):
    evec = pred - truth
    edist = np.linalg.norm(evec, axis=1)
    return evec, edist


def metrics_from_error(edist):
    return {
        'rmse_mm': float(np.sqrt(np.mean(edist ** 2))),
        'mean_positional_error_mm': float(np.mean(edist)),
        'p95_positional_error_mm': float(np.percentile(edist, 95)),
        'median_positional_error_mm': float(np.median(edist)),
        'max_positional_error_mm': float(np.max(edist)),
    }


def affine_cv_errors(mp_xy, ot_xz, folds=5):
    """Cross-validated affine residuals. Uses interleaved folds to cover the full trajectory in each fold."""
    n = len(mp_xy)
    pred = np.full_like(ot_xz, np.nan, dtype=float)
    idx = np.arange(n)
    for fold in range(folds):
        test = (idx % folds) == fold
        train = ~test
        if train.sum() < 10 or test.sum() == 0:
            continue
        B = fit_affine(mp_xy[train], ot_xz[train])
        pred[test] = apply_affine(mp_xy[test], B)
    valid = np.isfinite(pred).all(axis=1)
    return pred[valid], ot_xz[valid], valid



def estimate_global_lag(mp_pairs, ot_trials):
    """Estimate a session-level MP->OT clock offset from interval overlap only.

    Positive lag means MP timestamps are shifted forward to compare with OptiTrack.
    """
    best = None
    rows = []
    for lag in GLOBAL_LAG_SEARCH_SECONDS:
        score = 0.0
        matched_intervals = 0
        for pair in mp_pairs:
            shifted_start = pair['start'] + pd.Timedelta(seconds=float(lag))
            shifted_end = pair['end'] + pd.Timedelta(seconds=float(lag))
            for ot in ot_trials:
                overlap_s = max(0.0, (min(shifted_end, ot['end']) - max(shifted_start, ot['start'])).total_seconds())
                required_s = max(MIN_OVERLAP_SECONDS, MIN_OT_COVERAGE_FRACTION * ot['duration_s'])
                if overlap_s >= required_s:
                    matched_intervals += 1
                    score += overlap_s
        rows.append({'global_lag_s': float(lag), 'coverage_score_s': score, 'matched_intervals': matched_intervals})
        candidate = (score, matched_intervals, float(lag))
        if best is None or candidate > best:
            best = candidate
    lag_table = pd.DataFrame(rows).sort_values(['coverage_score_s', 'matched_intervals'], ascending=False)
    return best[2], lag_table


def compare_pair_to_ot(pair, ot_trial, global_lag_s, min_overlap_s=MIN_OVERLAP_SECONDS):
    """Compare one MP pair with one OT trial if shifted timestamps overlap enough."""
    shifted_pair_start = pair['start'] + pd.Timedelta(seconds=float(global_lag_s))
    shifted_pair_end = pair['end'] + pd.Timedelta(seconds=float(global_lag_s))
    shifted_overlap_start = max(shifted_pair_start, ot_trial['start'])
    shifted_overlap_end = min(shifted_pair_end, ot_trial['end'])
    shifted_overlap_s = (shifted_overlap_end - shifted_overlap_start).total_seconds()
    required_overlap_s = max(min_overlap_s, MIN_OT_COVERAGE_FRACTION * ot_trial['duration_s'])
    if shifted_overlap_s < required_overlap_s:
        return None

    mp_t_all = seconds(pair['time'])
    ot_t_all = seconds(ot_trial['time'])

    best = None
    for local_lag in LOCAL_LAG_SEARCH_SECONDS:
        effective_lag = float(global_lag_s + local_lag)
        # Positive effective lag means compare this MP sample to a later OT timestamp.
        shifted_mp_t = mp_t_all + effective_lag
        mask = (shifted_mp_t >= ot_t_all[0]) & (shifted_mp_t <= ot_t_all[-1])
        if mask.sum() < 50:
            continue
        covered_s = float(shifted_mp_t[mask].max() - shifted_mp_t[mask].min()) if mask.sum() > 1 else 0.0
        if covered_s < required_overlap_s:
            continue
        mp_xy = pair['mp_xy_px'][mask]
        ot_xz = interp_2d(shifted_mp_t[mask], ot_t_all, ot_trial['ot_xz_mm'])
        good = np.isfinite(mp_xy).all(axis=1) & np.isfinite(ot_xz).all(axis=1)
        mp_xy = mp_xy[good]
        ot_xz = ot_xz[good]
        shifted_used = shifted_mp_t[mask][good]
        original_used = mp_t_all[mask][good]
        if len(mp_xy) < 50:
            continue
        B = fit_affine(mp_xy, ot_xz)
        pred = apply_affine(mp_xy, B)
        _, edist = errors_from_predictions(pred, ot_xz)
        rmse = float(np.sqrt(np.mean(edist ** 2)))
        if best is None or rmse < best['lag_fit_rmse']:
            best = {
                'local_lag_s': float(local_lag),
                'effective_lag_s': effective_lag,
                'lag_fit_rmse': rmse,
                'mp_xy': mp_xy,
                'ot_xz': ot_xz,
                'mp_time_s': original_used,
                'shifted_time_s': shifted_used,
                'covered_s': covered_s,
            }

    if best is None:
        return None

    mp_xy = best['mp_xy']
    ot_xz = best['ot_xz']
    pred_cv, truth_cv, cv_valid = affine_cv_errors(mp_xy, ot_xz, CV_FOLDS)
    evec_cv, edist_cv = errors_from_predictions(pred_cv, truth_cv)
    metrics = metrics_from_error(edist_cv)
    cv_error_full = np.full(len(mp_xy), np.nan, dtype=float)
    cv_error_full[cv_valid] = edist_cv

    # Fit final affine on all samples for plotting and calibration diagnostics.
    B_all = fit_affine(mp_xy, ot_xz)
    pred_all = apply_affine(mp_xy, B_all)
    evec_all, edist_all = errors_from_predictions(pred_all, ot_xz)

    linear = B_all[:2, :].T  # shape 2x2 maps [px,py] to [X,Z]
    _, singular_values, _ = np.linalg.svd(linear)
    px_to_mm_major = float(singular_values[0])
    px_to_mm_minor = float(singular_values[-1])
    anisotropy = px_to_mm_major / px_to_mm_minor if px_to_mm_minor > 0 else np.nan

    sample_df = pd.DataFrame({
        'time_s_relative_to_segment': best['shifted_time_s'] - best['shifted_time_s'][0],
        'mp_original_time_s': best['mp_time_s'],
        'mp_shifted_to_ot_time_s': best['shifted_time_s'],
        'mp_x_px_smoothed': mp_xy[:, 0],
        'mp_y_px_smoothed': mp_xy[:, 1],
        'ot_X_mm': ot_xz[:, 0],
        'ot_Z_mm': ot_xz[:, 1],
        'mp_calibrated_X_mm': pred_all[:, 0],
        'mp_calibrated_Z_mm': pred_all[:, 1],
        'fit_error_mm_all_samples': edist_all,
        'cv_error_mm': cv_error_full,
    })
    seg_stem = f"{pair['pair']}_{pair['finger']}_{safe_stem(ot_trial['file'])}"
    sample_csv = SEGMENT_DIR / f'{seg_stem}_samples.csv'
    sample_df.to_csv(sample_csv, index=False)

    row = {
        'finger': pair['finger'],
        'mp_pair': pair['pair'],
        'ot_file': ot_trial['file'],
        'ot_marker': ot_trial['marker'],
        'shifted_overlap_seconds': shifted_overlap_s,
        'required_overlap_seconds': required_overlap_s,
        'covered_seconds_used': best['covered_s'],
        'samples_compared': int(len(edist_cv)),
        'global_lag_s_mp_to_ot': float(global_lag_s),
        'local_lag_s_mp_to_ot': best['local_lag_s'],
        'effective_lag_s_mp_to_ot': best['effective_lag_s'],
        **metrics,
        'fit_rmse_all_samples_mm': float(np.sqrt(np.mean(edist_all ** 2))),
        'mean_error_x_mm_all_samples': float(np.mean(evec_all[:, 0])),
        'mean_error_z_mm_all_samples': float(np.mean(evec_all[:, 1])),
        'affine_px_to_mm_major': px_to_mm_major,
        'affine_px_to_mm_minor': px_to_mm_minor,
        'affine_anisotropy_major_minor': float(anisotropy),
        'affine_b00_pxX_to_otX': float(B_all[0, 0]),
        'affine_b01_pxX_to_otZ': float(B_all[0, 1]),
        'affine_b10_pxY_to_otX': float(B_all[1, 0]),
        'affine_b11_pxY_to_otZ': float(B_all[1, 1]),
        'affine_tx_otX_mm': float(B_all[2, 0]),
        'affine_tz_otZ_mm': float(B_all[2, 1]),
        'segment_samples_csv': str(sample_csv),
    }

    return row, sample_df


In [ ]:
ot_trials = load_ot_big_trials()
mp_pairs = load_mp_pairs()

print(f'Loaded {len(ot_trials)} OptiTrack big trials')
print(f'Loaded {len(mp_pairs)} Mediapipe pairs')
print('\nMediapipe pairs:')
for p in mp_pairs:
    print(f"  {p['pair']} | finger={p['finger']} | {p['start']} -> {p['end']} | rows={p['rows']} | fs~{p['fs_est']:.1f} Hz")

print('\nOptiTrack big trial ranges:')
for o in ot_trials:
    print(f"  {o['file']} | marker={o['marker']} | {o['start']} -> {o['end']} | {o['duration_s']:.1f}s")

In [ ]:
global_lag_s, global_lag_table = estimate_global_lag(mp_pairs, ot_trials)
global_lag_table.to_csv(OUT_DIR / 'global_lag_search_table.csv', index=False)
print(f'Estimated global Mediapipe -> OptiTrack timestamp shift: {global_lag_s:.1f} s')
print('Positive means Mediapipe timestamps were shifted later to align with OptiTrack.\n')

segment_rows = []
segment_samples = []
all_overlap_candidates = []

for pair in mp_pairs:
    shifted_start = pair['start'] + pd.Timedelta(seconds=float(global_lag_s))
    shifted_end = pair['end'] + pd.Timedelta(seconds=float(global_lag_s))
    for ot in ot_trials:
        shifted_overlap_s = max(0.0, (min(shifted_end, ot['end']) - max(shifted_start, ot['start'])).total_seconds())
        required_s = max(MIN_OVERLAP_SECONDS, MIN_OT_COVERAGE_FRACTION * ot['duration_s'])
        if shifted_overlap_s > 0:
            all_overlap_candidates.append({
                'finger': pair['finger'],
                'mp_pair': pair['pair'],
                'ot_file': ot['file'],
                'shifted_overlap_seconds': shifted_overlap_s,
                'required_overlap_seconds': required_s,
                'used_for_metrics': shifted_overlap_s >= required_s,
                'global_lag_s': global_lag_s,
            })
        result = compare_pair_to_ot(pair, ot, global_lag_s)
        if result is not None:
            row, samples = result
            segment_rows.append(row)
            segment_samples.append((row, samples))

segment_metrics = pd.DataFrame(segment_rows)
overlap_table = pd.DataFrame(all_overlap_candidates).sort_values(['mp_pair', 'shifted_overlap_seconds'], ascending=[True, False])

if segment_metrics.empty:
    raise RuntimeError('No sufficient timestamp overlaps found between OptiTrack and Mediapipe data after global lag correction.')

# Aggregate per finger using sample-weighted raw error samples.
finger_rows = []
for finger, group in segment_metrics.groupby('finger'):
    errs = []
    total_samples = 0
    for _, r in group.iterrows():
        s = pd.read_csv(r['segment_samples_csv'])
        errs.append(s['cv_error_mm'].dropna().to_numpy(dtype=float))
        total_samples += len(s)
    errs = np.concatenate(errs)
    m = metrics_from_error(errs)
    finger_rows.append({
        'finger': finger,
        'segments_compared': len(group),
        'samples_compared': int(total_samples),
        **m,
        'global_lag_s': float(global_lag_s),
        'mean_effective_lag_s': float(group['effective_lag_s_mp_to_ot'].mean()),
        'mean_affine_px_to_mm_major': float(group['affine_px_to_mm_major'].mean()),
        'mean_affine_px_to_mm_minor': float(group['affine_px_to_mm_minor'].mean()),
    })

finger_order = {'index': 0, 'middle': 1, 'ring': 2, 'pinky': 3}
finger_metrics = pd.DataFrame(finger_rows)
finger_metrics['finger_order'] = finger_metrics['finger'].map(finger_order).fillna(99)
finger_metrics = finger_metrics.sort_values('finger_order').drop(columns=['finger_order'])

# OT files without sufficient matching camera data in this Mediapipe folder after global shift.
used_ot = set(segment_metrics['ot_file'])
unmatched = []
for ot in ot_trials:
    best_overlap = 0.0
    best_pair = ''
    best_finger = ''
    for pair in mp_pairs:
        shifted_start = pair['start'] + pd.Timedelta(seconds=float(global_lag_s))
        shifted_end = pair['end'] + pd.Timedelta(seconds=float(global_lag_s))
        overlap_s = max(0.0, (min(shifted_end, ot['end']) - max(shifted_start, ot['start'])).total_seconds())
        if overlap_s > best_overlap:
            best_overlap = overlap_s
            best_pair = pair['pair']
            best_finger = pair['finger']
    if ot['file'] not in used_ot:
        unmatched.append({
            'ot_file': ot['file'],
            'ot_start': ot['start'],
            'ot_end': ot['end'],
            'best_shifted_mp_pair': best_pair,
            'best_shifted_mp_finger': best_finger,
            'best_shifted_overlap_seconds': best_overlap,
            'reason': f'shifted overlap below {MIN_OT_COVERAGE_FRACTION:.0%} of OT duration / below {MIN_OVERLAP_SECONDS:g}s, or no matching data in provided Mediapipe folder',
        })
unmatched_ot = pd.DataFrame(unmatched)

segment_metrics.to_csv(OUT_DIR / 'segment_error_metrics.csv', index=False)
finger_metrics.to_csv(OUT_DIR / 'finger_error_metrics.csv', index=False)
overlap_table.to_csv(OUT_DIR / 'timestamp_overlap_table.csv', index=False)
unmatched_ot.to_csv(OUT_DIR / 'unmatched_optitrack_files.csv', index=False)

print('Compared segments:')
print(segment_metrics[['finger','mp_pair','ot_file','shifted_overlap_seconds','samples_compared','effective_lag_s_mp_to_ot','rmse_mm','mean_positional_error_mm','p95_positional_error_mm']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print('\nPer-finger aggregate:')
print(finger_metrics[['finger','segments_compared','samples_compared','rmse_mm','mean_positional_error_mm','p95_positional_error_mm','median_positional_error_mm']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"\nUnmatched OptiTrack files from this Mediapipe folder after lag correction: {len(unmatched_ot)}")

In [ ]:
# Per-segment overlay plots: OT path vs calibrated Mediapipe path.
for row, samples in segment_samples:
    stem = f"{row['mp_pair']}_{row['finger']}_{safe_stem(row['ot_file'])}"
    fig, ax = plt.subplots(figsize=(8, 7.5))
    ax.plot(samples['ot_X_mm'], samples['ot_Z_mm'], color='tab:blue', linewidth=2, label='OptiTrack X-Z')
    ax.plot(samples['mp_calibrated_X_mm'], samples['mp_calibrated_Z_mm'], color='tab:orange', linewidth=1.5, alpha=0.9, label='Mediapipe calibrated to X-Z')
    sc = ax.scatter(samples['ot_X_mm'], samples['ot_Z_mm'], c=samples['time_s_relative_to_segment'], s=5, cmap='viridis', alpha=0.45)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('X / East-West (mm)')
    ax.set_ylabel('Z / North-South (mm)')
    ax.set_title(f"{row['finger']} | {row['ot_file']}\nRMSE={row['rmse_mm']:.2f} mm, mean={row['mean_positional_error_mm']:.2f} mm, p95={row['p95_positional_error_mm']:.2f} mm, lag={row['effective_lag_s_mp_to_ot']:.2f}s")
    ax.legend(loc='best')
    fig.colorbar(sc, ax=ax, label='Time in compared segment (s)')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{stem}_overlay_XZ.png', bbox_inches='tight')
    plt.close(fig)

print(f'Wrote {len(segment_samples)} segment overlay plots to {PLOT_DIR}')

In [ ]:
# Summary bar plot per finger.
fig, ax = plt.subplots(figsize=(9, 5.5))
labels = finger_metrics['finger'].tolist()
x = np.arange(len(labels))
width = 0.25
ax.bar(x - width, finger_metrics['rmse_mm'], width, label='RMSE')
ax.bar(x, finger_metrics['mean_positional_error_mm'], width, label='Mean error')
ax.bar(x + width, finger_metrics['p95_positional_error_mm'], width, label='95th percentile')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Position error (mm)')
ax.set_title('OptiTrack vs Mediapipe calibrated 2D positional error by finger')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'finger_error_metrics_bar.png', bbox_inches='tight')
plt.close(fig)

# Error distributions.
fig, ax = plt.subplots(figsize=(9, 5.5))
for finger in finger_metrics['finger']:
    data = []
    for _, r in segment_metrics[segment_metrics['finger'].eq(finger)].iterrows():
        s = pd.read_csv(r['segment_samples_csv'])
        data.append(s['fit_error_mm_all_samples'].to_numpy(dtype=float))
    data = np.concatenate(data)
    ax.hist(data, bins=60, density=True, alpha=0.35, label=finger)
ax.set_xlabel('Calibrated 2D positional error (mm)')
ax.set_ylabel('Density')
ax.set_title('Error distribution by finger')
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'finger_error_distributions.png', bbox_inches='tight')
plt.close(fig)

print(PLOT_DIR / 'finger_error_metrics_bar.png')
print(PLOT_DIR / 'finger_error_distributions.png')

In [ ]:
# Write a concise interpretation report.
mp_start = min(p['start'] for p in mp_pairs)
mp_end = max(p['end'] for p in mp_pairs)
used_segments = len(segment_metrics)
unmatched_count = len(unmatched_ot)

lines = []
lines.append('# OptiTrack vs Mediapipe tracker comparison report')
lines.append('')
lines.append('## Scope')
lines.append(f'- Mediapipe capture range: `{mp_start}` to `{mp_end}`')
lines.append(f'- OptiTrack big trials loaded: {len(ot_trials)}')
lines.append(f'- OptiTrack/Mediapipe overlaps used: {used_segments}')
lines.append(f'- OptiTrack files without enough overlap in this Mediapipe folder: {unmatched_count}')
lines.append('')
lines.append('## Calibration / alignment')
lines.append('- Compared OptiTrack X-Z mm against Mediapipe active-finger image x-y pixels.')
lines.append('- For each timestamp-overlap segment, Mediapipe pixels were smoothed and mapped to OptiTrack X-Z using an affine calibration.')
lines.append('- The affine calibration accounts for zero offset, axis direction flips, rotation, scale, and mild camera anisotropy.')
lines.append(f'- Estimated global MP->OT clock shift: **{global_lag_s:.1f} s**; then a +/-2 s local lag search refined each segment.')
lines.append('- Metrics are 2D positional errors after calibration; this is not a full 3D error because the Mediapipe CSV has no depth coordinate.')
lines.append('')
lines.append('## Per-finger calibrated 2D error')
for _, r in finger_metrics.iterrows():
    lines.append(f"- **{r['finger']}**: RMSE={r['rmse_mm']:.2f} mm, mean={r['mean_positional_error_mm']:.2f} mm, 95th percentile={r['p95_positional_error_mm']:.2f} mm, segments={int(r['segments_compared'])}")
lines.append('')
lines.append('## Notes / limitations')
lines.append('- Only timestamp-overlapping OptiTrack files are compared. Later OptiTrack files are not matched to this Mediapipe folder because that would require assuming order without camera timestamps.')
lines.append('- The primary reported metrics use affine-calibrated 2D agreement. If absolute camera calibration is needed, a checkerboard/camera calibration or known mm-per-pixel target calibration should be used instead of fitting to OptiTrack.')
lines.append('- Large residuals can come from camera landmark error, OptiTrack marker/finger physical offset, synchronization lag, or real out-of-plane motion not visible to the overhead 2D camera.')
lines.append('')
lines.append('## Output files')
lines.append('- `segment_error_metrics.csv`')
lines.append('- `finger_error_metrics.csv`')
lines.append('- `timestamp_overlap_table.csv`')
lines.append('- `unmatched_optitrack_files.csv`')
lines.append('- `plots/*_overlay_XZ.png`')
lines.append('- `plots/finger_error_metrics_bar.png`')
lines.append('- `plots/finger_error_distributions.png`')

report = '\n'.join(lines)
(OUT_DIR / 'tracker_comparison_report.md').write_text(report, encoding='utf-8')
print(report)